In [1]:
def load_labels(path):
    labels = {}
    with open(path) as f:
        for line in f:
            img, lbl = line.strip().split(',')
            labels[img] = int(lbl)
    return labels


In [2]:
import json

with open("bbox_annot.json") as f:
    bbox_data = json.load(f)


In [3]:
import cv2
import numpy as np
from skimage.feature import graycomatrix, graycoprops

def extract_features(img_path, bbox):
    img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)

    x1, y1, x2, y2 = bbox
    roi = img[y1:y2, x1:x2]

    roi = cv2.resize(roi, (128, 128))

    glcm = graycomatrix(
        roi,
        distances=[1],
        angles=[0],
        levels=256,
        symmetric=True,
        normed=True
    )

    features = [
        graycoprops(glcm, 'contrast')[0, 0],
        graycoprops(glcm, 'homogeneity')[0, 0],
        graycoprops(glcm, 'energy')[0, 0],
        graycoprops(glcm, 'correlation')[0, 0]
    ]

    return features


In [ ]:
X, y = [], []

labels = load_labels("train.txt")

for img_name, lbl in labels.items():
    if img_name not in bbox_data:
        continue

    bbox = bbox_data[img_name]["bbs"][0][1]
    features = extract_features(f"imgs/{img_name}", bbox)

    X.append(features)
    y.append(lbl)

X = np.array(X)
y = np.array(y)


In [5]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

models = {
    "kNN": KNeighborsClassifier(n_neighbors=5),
    "SVM": SVC(kernel="rbf", probability=True),
    "Decision Tree": DecisionTreeClassifier(max_depth=5),
    "Random Forest": RandomForestClassifier(n_estimators=100)
}

for name, model in models.items():
    model.fit(X_train, y_train)
    preds = model.predict(X_test)

    print(f"\n{name}")
    print(classification_report(y_test, preds))



kNN
              precision    recall  f1-score   support

           0       0.57      0.70      0.63        80
           1       0.65      0.74      0.69       102
           2       0.38      0.11      0.17        45

    accuracy                           0.60       227
   macro avg       0.53      0.52      0.50       227
weighted avg       0.57      0.60      0.57       227


SVM
              precision    recall  f1-score   support

           0       0.62      0.64      0.63        80
           1       0.61      0.87      0.72       102
           2       0.00      0.00      0.00        45

    accuracy                           0.62       227
   macro avg       0.41      0.50      0.45       227
weighted avg       0.49      0.62      0.55       227


Decision Tree
              precision    recall  f1-score   support

           0       0.64      0.72      0.68        80
           1       0.67      0.73      0.69       102
           2       0.69      0.40      0.51       

c:\Users\lucky\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\metrics\_classification.py:1509: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
c:\Users\lucky\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\metrics\_classification.py:1509: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
c:\Users\lucky\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\metrics\_classification.py:1509: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, mo


Random Forest
              precision    recall  f1-score   support

           0       0.74      0.66      0.70        80
           1       0.69      0.79      0.74       102
           2       0.70      0.58      0.63        45

    accuracy                           0.70       227
   macro avg       0.71      0.68      0.69       227
weighted avg       0.71      0.70      0.70       227

